# 保留 NaN 的 Mixed Effects Random Forest：CU／AD／non-AD

以 `../ml/next_mmse_svm_rf_xgboost_CU_AD_nonAD.ipynb` 的資料、特徵及切分方法為基底；原檔保持不變。

模型：$Y_{ij}=f_{RF}(H_{ij})+b_i+\epsilon_{ij}$，$b_i\sim N(0,\tau^2)$、$\epsilon_{ij}\sim N(0,\sigma^2)$。第一版僅 random intercept，Z 是一欄 1；patient_id 只作 cluster。

不做 imputation、不標準化 X 或 y。預測保留原始 MMSE 分數，不裁切或四捨五入。日期只作排序與稽核；X 不含 target 日期或下一次實際間隔。

請使用專案 `.venv-merf` kernel 並 Run All。完整安裝與命令見 `README_merf_zh.md`。


In [ ]:
from pathlib import Path
import sys, logging
import pandas as pd
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data/processed').is_dir() and (p / 'notebooks').is_dir()), None)
if root is None:
    raise RuntimeError('請從專案內目錄啟動 notebook')
sys.path.insert(0, str(root / 'notebooks/build_models/mixed effect ml'))
from merf_workflow import (versions, save_json, synthetic_checks, prepare_data, run_group,
                           evaluation_tables, write_report, FEATURES, GROUPS, GRID, MAX_ITERATIONS)
logging.basicConfig(level=logging.WARNING)
out = root / 'results/merf_nan_CU_AD_nonAD'
out.mkdir(parents=True, exist_ok=True)
print(versions())
save_json(out / 'environment.json', versions())
print('固定 grid:', GRID, 'MERF iterations:', MAX_ITERATIONS)


## 人工資料：先驗證 NaN 相容性與防洩漏

包含部分缺值與整欄缺值，完整執行 RF／MERF fit／predict；驗證兩次量測無可校正 transition、未見病人 population 等於 fixed effect、m=0 校正等於 population、缺失最後 target 不回退，以及修改最終 MMSE 不改變 X／b_hat。

[MERF 1.0 原始碼](https://github.com/manifoldai/merf/blob/master/merf/merf.py) 直接把 X 傳給 estimator。使用原生 NaN RF；若環境不相容，本 cell 會拋出具體錯誤並停止，不偷偷填補或刪除缺值 X rows。不修改 site-packages。


In [ ]:
print(synthetic_checks(out))


## 資料檢查與處理紀錄

每列是 MMSE visit，附 CDR、已配對 CASI／血液與人口學資料。上游 `data_preprocessing.ipynb` 將 `MMSE(numerical)` 改名為 mmse、限制 0–30，未標準化。現有 CSV 沒有 next target；程式仍明確只由排序後 mmse 建立 y。

同日保留 CSV 第一列整列（不按 MMSE 選值，不合併欄位）。category 衝突整個病人排除；單一已知 category 可作病人 metadata。非數值、infinity、超範圍 MMSE 轉 NaN 並記錄；無效 ID／日期排除。缺失 MMSE visit 保留作歷史，缺失 target 排除。

上游血液使用 nearest 配對，可能找到未來檢驗。本流程將晚於歷史 visit 或日期不明的血液值與 gap 遮罩 NaN，不以其他檢驗填補；CASI 亦稽核日期。血液 gap 改成 log1p(visit_date − blood_date)，第一筆 visit gap 保持 NaN。這項資料可用時間與 gap 方向修正會提高部分缺失率，影響均有紀錄。


In [ ]:
csv = root / 'data/processed/model_df_next_mmse3.csv'
if not csv.exists():
    save_json(out / 'execution_status.json', {'synthetic':'passed','real_data':'not_run_missing_csv'})
    raise FileNotFoundError(f'人工檢查通過，尚未執行真實資料實驗：{csv}')
data, audit = prepare_data(csv, out)
print('資料列／病人:', audit['input_rows'], audit['clean_rows'], audit['clean_patients'])
print('既有 next 欄位:', audit['existing_next_columns'])
print('MMSE 範圍:', audit['mmse_observed_range'])
print('未來或日期不明檢驗遮罩:', audit['future_auxiliary_masked'])
display(data[FEATURES].isna().mean().rename('缺失比例').to_frame())


## 切分、樣本語意與 smoke run

沿用原 notebook 數值 ID 排序、drop_duplicates 順序及兩次 train_test_split（0.30、0.50，seed=42），先按病人分割 70/15/15 再建立樣本。清理後 eligible IDs 可能改變，實際 split IDs 完整保存。

v1→v2、v1–v2→v3、v1–v3→v4 全部用於 training；validation/test 每人只取最後 visit，缺失最後 MMSE 不回退。每欄有 last、mean、std(ddof=0)、min、max、observed_count、missing_rate。last 缺失不向前找；摘要只用非缺值，全缺值保持 NaN，單一有效值 std=0。

每筆保存 patient_id、history_end_date、target_date、target_visit_index（1-based）、history_n_visits；X 包含歷史跨度及 visit_gap_log。模型固定效果特徵與 original MMSE、CDR、CASI、血液、clinical 類別相同。先完成三組小型 RF／2 次 MERF iterations smoke，再正式調參。


In [ ]:
for group in GROUPS:
    run_group(data, group, out / 'smoke' / group, smoke=True)
print('三組 smoke run 通過')


## 調參、train+validation 重訓與個人校正

RF 與 MERF 使用相同 train rows、X，以各自 validation RMSE 選參數。MERF **主要情境固定為 history-calibrated**。Grid：200 trees、depth=10、leaf=3 或 5，seed=42、固定 20 iterations。test 不參與參數／迭代／校正方法選擇。選後以 train+validation 全部有效 transitions 重新訓練。

test 病人從未進入 MERF 訓練。僅用最終 target 之前已知歷史 transition 得到 $r_{ik}=y_{ik}-f_{RF}(H_{ik})$，Gaussian random-intercept 條件均值為

$$\hat b_i=\frac{\tau^2}{\sigma^2+m_i\tau^2}\sum_k r_{ik}.$$

變異數從訓練模型取得。第一筆 MMSE 不能當無歷史模型的校正反應；兩次量測預測第二次時 m=0、b=0。校正資料先移除最終 target 再另行建立。tau²／分母接近零時回到 population，sigma² 接近零另記退化狀態。

校正依 Gaussian random-intercept 與殘差獨立假設；重疊歷史的 transitions 可能相關。GLL 與變異數全程保存，固定 20 iterations 到限不等於收斂。原生 MERF population predict 對未見病人必須與 fixed effect 一致。


In [ ]:
results = []
for group in GROUPS:
    results.append(run_group(data, group, out / group))


## 評估、檢查與交付

test 預測列完全對齊。Last-MMSE 嚴格使用最近 visit 的 MMSE，缺失時另報覆蓋率；common_baseline_available 是所有模型的公平共同子集。另按 m=0／1／2+ 分層。MAE、RMSE、R²、MAPE(%) 皆原始尺度；MAPE 排除 y=0 並記錄數量，少於兩筆時 R² 為 NaN。

各組保存 split IDs、樣本索引／排除紀錄、X 缺失率、grid、選定參數、iteration history、group_report 與逐病人預測。總表與中文報告位於 results/merf_nan_CU_AD_nonAD。每組真實資料亦執行最終 target 擾動檢查，驗證 X／b_hat 不變、y／Z 有限、病人切分無交集、history_end_date < target_date、預測及變異數有限。


In [ ]:
predictions = pd.concat([r[0] for r in results], ignore_index=True)
metrics_table = evaluation_tables(predictions)
predictions.to_csv(out / 'test_predictions.csv', index=False)
metrics_table.to_csv(out / 'metrics.csv', index=False)
write_report(out, audit, [r[1] for r in results], metrics_table)
save_json(out / 'execution_status.json', {'synthetic':'passed','smoke':'passed','real_data':'completed'})
display(metrics_table[(metrics_table.scope == 'all') & (metrics_table.calibration_stratum == 'all')])
display(pd.DataFrame([r[1] for r in results]).drop(columns='checks'))
display(predictions.head())
print('輸出:', out)
print((out / 'report_zh.md').read_text(encoding='utf-8'))
